# IF29 — Analyse exploratoire des données (EDA)

## Projet : Détection de profils atypiques sur X (Twitter)

Ce notebook correspond à l'**Étape 2** du projet IF29 : comprendre les données avant toute modélisation ML.

**Questions guidées de l'EDA :**
- Y a-t-il des valeurs manquantes ?
- Quelle est la distribution de chaque variable ?
- Existe-t-il des outliers (valeurs extrêmes) ?
- Les variables sont-elles corrélées entre elles ?
- Quelles features engineered peuvent aider à détecter les profils atypiques ?

**Fichiers utilisés :**
- `raw/` — tweets bruts (581 fichiers JSONL, ~1,16 M tweets)
- `users_aggregated.csv` — profils agrégés MongoDB (sans label)
- `users_labeled_manual.csv` — même fichier + `label` + `anomaly_score` (ML commun)


## 1. Définition du profil atypique

> *« Un profil atypique est un profil dont les caractéristiques statistiques ou comportementales s'écartent significativement de la population moyenne. »*

Inspirée de Ferrara et al. (2016), Chu et al. (2012), Varol et al. (2017) et du framework SPOT :

| Dimension | Indicateurs | Signal d'anomalie |
|-----------|-------------|-------------------|
| Graphe social | followers, friends, ratio | Ratio extrême |
| Activité | nb_tweets, retweet_ratio | Amplification / burst |
| Contenu | hashtags, URLs, mentions | Spam / surcharge |
| Temporel | active_days, tweet_frequency | Activité concentrée |
| Visibilité (SPOT) | followers x retweet_ratio | Portée d'amplification |

Labellisation manuelle Excel : atypique si >= 2 critères sur 4 — voir `docs/LABELISATION.md`


In [ ]:
# Imports et configuration
import os
import glob
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

RAW_DIR = "raw"
DATA_AGG = "users_aggregated.csv"
DATA_LAB = "users_labeled_manual.csv"
RANDOM_STATE = 42

print("Bibliothèques chargées")


## 2. Données brutes — vue d'ensemble


In [ ]:
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.json")))
print(f"Nombre de fichiers JSONL : {len(raw_files)}")

sample_tweets = []
with open(raw_files[0], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        sample_tweets.append(json.loads(line))

tweet = sample_tweets[0]
print(f"Structure tweet : {list(tweet.keys())[:10]}")
print(f"Auteur (user) : {list(tweet['user'].keys())[:8]}")
print(f"Est un retweet : {'retweeted_status' in tweet}")

pd.DataFrame([{
    "user_id": t["user"]["id"],
    "screen_name": t["user"]["screen_name"],
    "text_len": len(t.get("text", "")),
    "is_retweet": "retweeted_status" in t,
} for t in sample_tweets])


**Pipeline Étape 1 :** tweets bruts -> enrichissement -> agrégation MongoDB par `user.id` -> `users_aggregated.csv`

**Hypothèse fondamentale :** seul l'auteur du tweet (`user`) est analysé, pas `retweeted_status.user`.


## 3. Chargement et aperçu du dataset agrégé


In [ ]:
df = pd.read_csv(DATA_AGG)

print(f"Dataset chargé : {df.shape[0]:,} utilisateurs, {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")
df.head()


In [ ]:
print("=" * 60)
print("TYPES ET VALEURS MANQUANTES")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("STATISTIQUES DESCRIPTIVES")
print("=" * 60)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[num_cols].describe().T.round(2)


## 4. Valeurs manquantes


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Valeurs manquantes": missing,
    "Pourcentage (%)": missing_pct
}).sort_values("Pourcentage (%)", ascending=False)

print("Valeurs manquantes par colonne :")
display(missing_df[missing_df["Valeurs manquantes"] > 0])

if missing_df["Valeurs manquantes"].sum() == 0:
    print("\nAucune valeur manquante sur les variables numériques.")
else:
    missing_pct[missing_pct > 0].plot(kind="bar", color="coral", edgecolor="black")
    plt.title("Pourcentage de valeurs manquantes par colonne")
    plt.ylabel("% manquant")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 5. Distribution des variables principales

Sur Twitter, la plupart des utilisateurs ont peu de followers, mais quelques-uns en ont énormément.
Distribution asymétrique (loi de puissance), classique sur les réseaux sociaux.
On utilise `log1p(x) = log(1+x)` pour mieux visualiser.


In [ ]:
cols_to_plot = [
    "followers_count", "friends_count", "nb_tweets", "nb_retweets",
    "retweet_ratio", "avg_hashtags", "avg_mentions", "followers_friends_ratio"
]
cols_to_plot = [c for c in cols_to_plot if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    data = np.log1p(df[col].clip(lower=0).dropna())
    axes[i].hist(data, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    axes[i].set_title(f"{col}\n(échelle log)", fontsize=10)
    axes[i].set_xlabel("log(1 + valeur)")

for j in range(len(cols_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution des variables principales (échelle logarithmique)", fontweight="bold")
plt.tight_layout()
plt.show()

print("INTERPRETATION :")
print("- Distributions tres asymetriques (queue longue a droite)")
print("- Quelques comptes ont des valeurs enormes (celebrites / bots)")
print("- log1p ici pour la visualisation ; StandardScaler seul avant ML (section 12)")


In [ ]:
box_cols = ["followers_count", "friends_count", "nb_tweets", "retweet_ratio", "avg_hashtags", "active_days"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, col in zip(axes, box_cols):
    data = df[col].clip(upper=df[col].quantile(0.95))
    ax.boxplot(data, vert=True)
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])
plt.suptitle("Boxplots (95e percentile) — presence d'outliers", fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Matrice de corrélation


In [ ]:
df_num = df.copy()
df_num["verified"] = df_num["verified"].astype(int)
df_num["default_profile_image"] = df_num["default_profile_image"].astype(int)

numeric_cols = df_num.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_num[numeric_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, fmt=".2f",
    cmap="RdYlGn", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True
)
plt.title("Matrice de corrélation entre les variables", fontweight="bold")
plt.tight_layout()
plt.show()

print("\nPaires fortement corrélées (|r| > 0.5) :")
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]} : r = {r:.3f}")


## 7. Feature engineering

Création de nouvelles variables pour capturer les comportements suspects (Chu et al., 2012 ; Ferrara et al., 2016 ; SPOT).

**Note :** `avg_favorites` et `avg_retweet_count` sont nulles sur ce dataset (collecte partielle).


In [ ]:
df_features = df.copy()

df_features["activity_rate"] = (
    df_features["nb_tweets"] / (df_features["active_days"] + 1)
).clip(0, 1)

df_features["engagement_score"] = (
    df_features["avg_favorites"] + df_features["avg_retweet_count"]
)

df_features["aggressiveness_score"] = (
    df_features["activity_rate"] * np.log1p(df_features["friends_count"])
)

df_features["visibility_score"] = (
    np.log1p(df_features["followers_count"]) * (1 + df_features["retweet_ratio"])
)

df_features["amplification_index"] = (
    df_features["nb_retweets"] / (df_features["avg_retweet_count"] + 1)
)

df_features["content_richness"] = (
    df_features["avg_hashtags"] + df_features["avg_urls"] + df_features["avg_mentions"]
)

df_features["is_verified"] = df_features["verified"].astype(int)

engineered = [
    "activity_rate", "aggressiveness_score", "visibility_score",
    "amplification_index", "content_richness", "engagement_score"
]
print("Features engineered calculées :")
for f in engineered:
    print(f"  {f:25s}  médiane={df_features[f].median():.3f}  max={df_features[f].max():.3f}")


In [ ]:
features_to_viz = [
    "followers_friends_ratio", "retweet_ratio", "activity_rate",
    "aggressiveness_score", "visibility_score", "amplification_index", "content_richness"
]
features_to_viz = [f for f in features_to_viz if f in df_features.columns]

ncols = 3
import math
nrows = math.ceil(len(features_to_viz) / ncols)  # nombre de lignes de subplots
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

for i, feat in enumerate(features_to_viz):
    data = df_features[feat].replace([np.inf, -np.inf], np.nan).dropna()
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    axes[i].hist(data.clip(p1, p99), bins=50, color="teal", edgecolor="white", alpha=0.8)
    axes[i].set_title(feat, fontsize=11, fontweight="bold")

for j in range(len(features_to_viz), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution des features engineered (tronquées P1-P99)", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
sample = df_features[["aggressiveness_score", "visibility_score"]].dropna()
sample = sample[np.isfinite(sample).all(axis=1)]
sample = sample.sample(min(5000, len(sample)), random_state=RANDOM_STATE)

plt.figure(figsize=(10, 7))
plt.scatter(sample["aggressiveness_score"], sample["visibility_score"],
            alpha=0.3, s=10, c="steelblue")
plt.xlabel("Aggressiveness Score (adapté SPOT)")
plt.ylabel("Visibility Score (adapté SPOT)")
plt.title("Scatter : Agressivité vs Visibilité", fontweight="bold")

q90_agg = sample["aggressiveness_score"].quantile(0.9)
q90_vis = sample["visibility_score"].quantile(0.9)
plt.axhline(q90_vis, color="red", linestyle="--", alpha=0.6, label="90e percentile visibilité")
plt.axvline(q90_agg, color="orange", linestyle="--", alpha=0.6, label="90e percentile agressivité")
plt.legend()
plt.tight_layout()
plt.show()

n_suspects = ((sample["aggressiveness_score"] > q90_agg) &
              (sample["visibility_score"] > q90_vis)).sum()
print(f"Profils quadrant suspect (agg. ET vis. > P90) : {n_suspects:,} ({n_suspects/len(sample)*100:.2f} %)")


## 8. Profils extrêmes — méthode IQR


In [ ]:
def count_outliers(series, factor=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_cols = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "retweet_ratio", "aggressiveness_score", "visibility_score"
]
outlier_stats = pd.DataFrame({
    "Outliers (IQR)": [count_outliers(df[c]) for c in outlier_cols],
    "Pct (%)": [count_outliers(df[c]) / len(df) * 100 for c in outlier_cols],
    "P99": [df[c].quantile(0.99) for c in outlier_cols],
    "Médiane": [df[c].median() for c in outlier_cols],
}, index=outlier_cols)
display(outlier_stats.round(2))

print("\nTop 5 — ratio followers/friends le plus élevé :")
display(df.nlargest(5, "followers_friends_ratio")[
    ["screen_name", "followers_count", "friends_count", "followers_friends_ratio", "nb_tweets", "retweet_ratio"]
])


In [ ]:
sample = df.sample(n=5000, random_state=RANDOM_STATE)
plt.figure(figsize=(10, 6))
plt.scatter(
    sample["followers_count"].clip(upper=sample["followers_count"].quantile(0.99)),
    sample["nb_tweets"], alpha=0.3, s=10, c="steelblue"
)
plt.xlabel("followers_count (99e pct)")
plt.ylabel("nb_tweets")
plt.title("Followers vs Activité (n=5 000)", fontweight="bold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Labellisation manuelle

Fichier produit : `users_labeled_manual.csv` (= `users_aggregated.csv` + `label` + `anomaly_score`)

| # | Règle | Condition |
|---|-------|----------|
| 1 | Amplification | retweet_ratio >= 0,8 |
| 2 | Spam | avg_urls >= 1,5 OU avg_mentions >= 2 |
| 3 | Burst | active_days = 1 ET nb_tweets >= 2 |
| 4 | Déséquilibre | followers_friends_ratio >= 30 |

Label atypique si >= 2 critères cumulés. Voir `docs/LABELISATION.md`.


## 10. Comparaison Normal vs Atypique


In [ ]:
df_lab = pd.read_csv(DATA_LAB)
df_lab["verified"] = df_lab["verified"].astype(int)

print(f"Dataset labelisé : {len(df_lab):,} profils")
print(df_lab["label"].value_counts())
print(f"Proportion atypiques : {df_lab['label'].mean()*100:.1f} %")

compare_cols = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "retweet_ratio", "avg_hashtags", "avg_mentions",
    "avg_urls", "active_days", "tweet_frequency"
]
comparison = df_lab.groupby("label")[compare_cols].mean()
comparison.index = ["Normal (0)", "Atypique (1)"]
comparison.T.round(2)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, compare_cols):
    d0 = df_lab[df_lab["label"] == 0][col].clip(upper=df_lab[col].quantile(0.99))
    d1 = df_lab[df_lab["label"] == 1][col].clip(upper=df_lab[col].quantile(0.99))
    ax.hist(d0, bins=40, alpha=0.6, label="Normal", color="#2196F3", density=True)
    ax.hist(d1, bins=40, alpha=0.6, label="Atypique", color="#F44336", density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("Distributions Normal vs Atypique", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 11. ACP — réduction dimensionnelle

Sélection du nombre de composantes par **seuil de variance cumulée** (pas la méthode du coude K-Means).

| Critère | Résultat typique (16 features) |
|---------|----------------------------------|
| Kaiser (valeur propre > 1) | 5 composantes |
| Seuil 70 % | 6 composantes |
| **Seuil 75 % (retenu)** | **7 composantes (~79 %)** |
| Seuil 80 % | 8 composantes |


In [ ]:
FEATURES_ML = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "nb_retweets", "retweet_ratio",
    "avg_tweet_length", "avg_hashtags", "avg_urls", "avg_mentions",
    "avg_favorites", "avg_retweet_count",
    "active_days", "tweet_frequency", "verified", "default_profile_image"
]

df["verified"] = df["verified"].astype(int)
df["default_profile_image"] = df["default_profile_image"].astype(int)
X_full = StandardScaler().fit_transform(df[FEATURES_ML].astype(float))

pca_full = PCA().fit(X_full)
evr = pca_full.explained_variance_ratio_
var_cum = np.cumsum(evr) * 100
eigenvalues = pca_full.explained_variance_

n_kaiser = int(np.sum(eigenvalues > 1))
n_70 = int(np.searchsorted(var_cum, 70) + 1)
N_COMPONENTS = int(np.searchsorted(var_cum, 75) + 1)  # seuil 75 % retenu
n_80 = int(np.searchsorted(var_cum, 80) + 1)

print("Sélection composantes ACP (16 features, full dataset):")
print(f"  Kaiser (>1)  : {n_kaiser}")
print(f"  Seuil 70 %   : {n_70} ({var_cum[n_70-1]:.1f} %)")
print(f"  Seuil 75 %   : {N_COMPONENTS} ({var_cum[N_COMPONENTS-1]:.1f} %)  <- RETENU")
print(f"  Seuil 80 %   : {n_80} ({var_cum[n_80-1]:.1f} %)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1, len(evr)+1), evr*100, "bo-")
axes[0].set_title("Scree plot")
axes[0].set_xlabel("Composante")
axes[0].set_ylabel("Variance (%)")
axes[0].grid(True)

axes[1].plot(range(1, len(var_cum)+1), var_cum, "bo-")
axes[1].axhline(70, color="r", linestyle="--", label="70 %")
axes[1].axhline(80, color="g", linestyle="--", label="80 %")
axes[1].axvline(N_COMPONENTS, color="orange", linestyle=":", linewidth=2,
                label=f"Retenu={N_COMPONENTS}")
axes[1].set_title("Variance cumulée")
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée (%)")
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

# Visualisation 2D (échantillon labelisé)
sample_lab = df_lab.sample(n=10000, random_state=RANDOM_STATE)
X_sample = StandardScaler().fit_transform(sample_lab[FEATURES_ML].astype(float))
X_pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_sample)

fig, ax = plt.subplots(figsize=(10, 7))
for label, color, name in [(0, "#2196F3", "Normal"), (1, "#F44336", "Atypique")]:
    mask = sample_lab["label"].values == label
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], alpha=0.3, s=8, c=color, label=name)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("ACP 2D — Normal vs Atypique (visualisation, n=10 000)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n=> {N_COMPONENTS} composantes retenues pour le ML (~{var_cum[N_COMPONENTS-1]:.1f} % variance)")
print("(Distinct du k=7 clusters K-Means — section non supervisée)")


## 12. Normalisation (aperçu)

Même prétraitement que les notebooks ML (`Groupe3_profils_atypiques_non_Sup.ipynb` et `Groupe3_profils_atypiques_Sup.ipynb`) :

- **16 features brutes** (`FEATURES_ML`) — `label` et `anomaly_score` exclus
- **`StandardScaler`** : z = (x − mean) / std


In [ ]:
df_lab["default_profile_image"] = df_lab["default_profile_image"].astype(int)

X = df_lab[FEATURES_ML].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES_ML)

print("Données normalisées — Shape :", X_scaled.shape)
print("\nVérification (mean ≈ 0, std ≈ 1) :")
X_scaled_df.describe().T[["mean", "std"]].round(3).head(8)


## 13. Tableau récapitulatif des features ML


In [ ]:
recap = pd.DataFrame([
    {"Feature": "followers_count",         "Type": "Brute",      "Source": "MongoDB",        "Justification": "Popularité (Chu 2012)"},
    {"Feature": "friends_count",           "Type": "Brute",      "Source": "MongoDB",        "Justification": "Activité de following"},
    {"Feature": "followers_friends_ratio", "Type": "Brute",      "Source": "MongoDB",        "Justification": "Discriminant humain/bot"},
    {"Feature": "nb_tweets",               "Type": "Brute",      "Source": "MongoDB",        "Justification": "Activité période"},
    {"Feature": "retweet_ratio",           "Type": "Brute",      "Source": "MongoDB",        "Justification": "Amplification (Ferrara 2016)"},
    {"Feature": "avg_hashtags",            "Type": "Brute",      "Source": "MongoDB",        "Justification": "Contenu / spam"},
    {"Feature": "active_days",             "Type": "Brute",      "Source": "MongoDB",        "Justification": "Comportement temporel"},
    {"Feature": "activity_rate",           "Type": "Engineered", "Source": "SPOT (adapté)",  "Justification": "Activité concentrée"},
    {"Feature": "aggressiveness_score",    "Type": "Engineered", "Source": "SPOT (adapté)",  "Justification": "Following agressif"},
    {"Feature": "visibility_score",        "Type": "Engineered", "Source": "SPOT (adapté)",  "Justification": "Portée d'amplification"},
    {"Feature": "amplification_index",     "Type": "Engineered", "Source": "Ferrara 2016",   "Justification": "Asymétrie RT émis/reçus"},
    {"Feature": "content_richness",        "Type": "Engineered", "Source": "EDA",            "Justification": "Hashtags + URLs + mentions"},
])

print("=" * 80)
print("TABLEAU RECAPITULATIF — FEATURES")
print("=" * 80)
display(recap)


## 14. Synthèse et prochaines étapes

**Observations clés :**
1. 643 124 profils, distributions asymétriques
2. Peu de valeurs manquantes, engagement nul (collecte partielle)
3. Features SPOT/Ferrara : exploration uniquement (non utilisées en ML)
4. Labels manuels : 16,9 % atypiques
5. ACP : **7 composantes** (seuil 75 %, ~79 %) pour le non supervisé

| Notebook | Fichier | ACP |
|----------|---------|-----|
| `Groupe3_profils_atypiques_non_Sup.ipynb` | `users_labeled_manual.csv` | 7 comp. (16 feat.) |
| `Groupe3_profils_atypiques_Sup.ipynb` | `users_labeled_manual.csv` | 5 comp. (8 feat.) |
| `Groupe3_profils_atypiques_Final.ipynb` | Synthèse | — |

In [ ]:
recap_eda = pd.DataFrame({
    "Métrique": [
        "Fichiers bruts JSONL", "Profils agrégés", "Features ML",
        "Comptes vérifiés", "Médiane followers", "Médiane nb_tweets",
        "Profils atypiques (labels)", "Variance ACP non sup. (7 comp.)",
        "Quadrant suspect SPOT (P90)"
    ],
    "Valeur": [
        f"{len(raw_files)}",
        f"{len(df):,}",
        f"{len(FEATURES_ML)}",
        f"{df['verified'].mean()*100:.2f} %",
        f"{df['followers_count'].median():,.0f}",
        f"{df['nb_tweets'].median():,.0f}",
        f"{df_lab['label'].mean()*100:.1f} %",
        f"{var_cum[6]:.1f} %",
        f"{n_suspects:,} ({n_suspects/len(sample)*100:.2f} %)"
    ]
})
print("=" * 50)
print(" RECAPITULATIF EDA")
print("=" * 50)
display(recap_eda)
